In [78]:
from skimage.color import label2rgb
from scipy.ndimage import zoom
from skimage.transform import resize
import matplotlib.pyplot as plt
from os import getcwd
from PIL.ImageColor import getcolor

from tifffile import imread

import numpy as np


examples = ["Frame5_4.6x.tiff", # lighter
            "Box13_P1_JK_4600x_0118.tiff", # weird shape
            "Box13_P1_JK_4600x_0103.tiff", 
            "Box13_P1_JK_4600x_0099.tiff",
            "Box13_O2_JK_4600x_0004.tiff", # darker
            ]

PREFIX = ""
MASK_SUFFIX = "_segmentation.tifnomalized.tif"

#PRED_FOLDERS = ["masks", "preds/dv2_out", "preds/rf_out_no_crf"]
# PRED_FOLDERS = ["masks", "preds/classical_crf", "preds/FeatUp_crf", "preds/bilinear_trained_sigma_0_crf", "preds/DINO-S-8_crf", "preds/DINOv2-S-14_crf", "preds/hybrid_crf"]
# PRED_FOLDERS = ["masks", "preds/classical_", "preds/FeatUp_crf", "preds/bilinear_trained_sigma_0_crf", "preds/DINO-S-8_crf", "preds/DINOv2-S-14_crf", "preds/hybrid_crf"]
PRED_FOLDERS = ["masks", "preds/classical_", "preds/bilinear_trained_sigma_0_", "preds/hybrid_"]
# PRED_FOLDERS = ["masks", "preds/classical_", "preds/hybrid_crf"]

COLOURS = [
    "#ffffff",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#648FFF",
    "#FFB000",
    "#5E503F",
    "#DFD9E2"
]


COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]

plt.style.use("thesis.mplstyle")

def remap_label_arr(arr: np.ndarray) -> np.ndarray:
    unq_vals = sorted(np.unique(arr))
    #div_val = unq_vals[1] if unq_vals[0] == 0 else unq_vals[0]
    div_val = unq_vals[-1] - unq_vals[-2]
    return (arr // div_val) - 1

def swap_arr_vals(arr: np.ndarray, mapping: dict[int, int]) -> np.ndarray:
    out = np.zeros_like(arr)
    for k, v in mapping.items():
        out[arr == k] = v
    return out

In [79]:
from PIL import Image

def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

def apply_labels_as_overlay(labels: np.ndarray, img: Image.Image, colors: list, alpha: float=1.0) -> Image.Image:
    labels_unsqueezed = np.expand_dims(labels, -1)

    overlay = label2rgb(labels, colors=colors, kind='overlay', bg_label=0, image_alpha=1, alpha=alpha)
    out = np.where(labels_unsqueezed, overlay * 255, np.array(img)).astype(np.uint8)
    img_with_labels = Image.fromarray(out)
    return img_with_labels

def add_line_annotation_between_axes(ax1, ax2, color='black', lw=1, dx: float=0.5):
    fig = ax1.figure
    transFigure = fig.transFigure.inverted()

    ax1_bot_y = ax1.get_position().y0
    ax1_left_x = ax1.get_position().x0
    ax2_top_y = ax2.get_position().y1

    mid_y = (ax1_bot_y + ax2_top_y) / 2

    line = plt.Line2D((ax1_left_x, ax1_left_x + dx), (mid_y, mid_y), transform=fig.transFigure, color='#bababa', lw=lw, dash_capstyle='round', solid_capstyle='round')
    fig.lines.append(line)


    # coord1 = transFigure.transform(ax1.transAxes.transform((1, 0.5)))
    # coord2 = transFigure.transform(ax2.transAxes.transform((0, 0.5)))
    # line = plt.Line2D((coord1[0], coord2[0]), (     coord1[1], coord2[1]), transform=fig.transFigure, color=color, lw=lw)
    # fig.lines.append(line) 

In [80]:

train_stack = imread("training_data/train_stack_small.tif")
greyscale_labels = imread("training_data/wss_train_labels.tiff").astype(np.int32)

# print(greyscale_labels.shape, greyscale_labels.dtype)

# color_list = [[255, 255, 255,], [31, 119, 180], [255, 127, 14],  [44, 160, 44]]
# COLORS = np.array(color_list) / 255.0

plus_label_imgs = []
for i, arr in enumerate(train_stack):
    greyscale_label = greyscale_labels[i]
    label = remap_label_arr(greyscale_label)
    if i == 5:
        label[0, 0] = 0
    label = swap_arr_vals(label, {0: 2, 1:1, 2:0,-1:-1})

    plus_label_img = apply_labels_as_overlay(label + 1, Image.fromarray(arr).convert('RGB'), colors=COLORS[1:], alpha=1)
    # img = label2rgb(label, arr, colors=COLORS[1:], bg_label=-1, alpha=0.4)
    plus_label_imgs.append(plus_label_img)

In [81]:

fig, axs = plt.subplots(nrows=len(PRED_FOLDERS) + 2, ncols=len(examples))

fig.set_size_inches(7, 2.3 * 3)
plt.rcParams["font.family"] = "serif"
#titles = ["Ground Truth", "Deep Features", "Classical Features"]
titles = ["Ground Truth", "Classical", "Bilinear", "Hybrid"]
#titles = ["Ground Truth", "Classical", "Ours (Hybrid)"]

for col, fname in enumerate(examples):
    original = imread(f"data/{fname[:-1]}")

    img_ax = axs[1, col]
    img_ax.imshow(original, cmap='binary_r')

    labels_ax = axs[0, col]
    labels_ax.imshow(plus_label_imgs[col])

    if col == 0:
        labels_ax.set_ylabel("Train Images")

    
    hide_axes(img_ax)
    hide_axes(labels_ax)

    if col == 0:
        add_line_annotation_between_axes(labels_ax, img_ax, color='black', lw=1, dx=0.75)
        add_line_annotation_between_axes(axs[2, col], axs[3, col], color='black', lw=1, dx=0.75)


    if col == 0:
        img_ax.set_ylabel("Test Images")
    

    for row, pred_folder in enumerate(PRED_FOLDERS):
        ax = axs[row + 2, col]

        if (col == 0):
            ax.set_ylabel(titles[row])

        suffix = MASK_SUFFIX if row == 0 else ""
        
        if row == 0:
            real_fname = fname[:-1]
        else:
            real_fname = fname

        low_res_data = imread(f"{pred_folder}/{real_fname}{suffix}")
        low_res_data = remap_label_arr(low_res_data)
        
        data = resize(low_res_data, (1024, 1024), preserve_range=True) if row > 0 else low_res_data
        data = data.astype(np.uint8)
        overlay = label2rgb(data + 1, None, colors=COLORS[1:], alpha=1, bg_label=0)
        
        ax.imshow(overlay)
        ax.set_xticks([])
        ax.set_yticks([])
        #ax.set_axis_off()
# plt.tight_layout()

# name = 'full_crf'

SAVE = True
if SAVE:
    plt.savefig("../../notebooks/figures/fig_pdf/tt_wss_preds.pdf", bbox_inches="tight")
    plt.close()

In [82]:
%%capture
train_stack = imread("training_data/train_stack_small.tif")
greyscale_labels = imread("training_data/wss_train_labels.tiff").astype(np.int32)

# print(greyscale_labels.shape, greyscale_labels.dtype)

# color_list = [[255, 255, 255,], [31, 119, 180], [255, 127, 14],  [44, 160, 44]]
# COLORS = np.array(color_list) / 255.0

r, c = 2, 3
fig, axs = plt.subplots(nrows=r, ncols=c)
fig.set_size_inches(24, 16)
for i, arr in enumerate(train_stack):
    x, y = i % c, i // c
    ax = axs[y, x]
    greyscale_label = greyscale_labels[i]
    label = remap_label_arr(greyscale_label)
    print(np.unique(label))
    if i == 5:
        label[0, 0] = 0
    #    label  = np.where(label == 0, 0, labels_arr[-1])
    img = label2rgb(label, arr, colors=COLORS[1:], bg_label=-1, alpha=0.4)
    ax.imshow(img)

    ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"figures/labels.png", bbox_inches="tight")

In [83]:
data = np.genfromtxt("figures/scaling.csv", delimiter=",")[1:]
x = data[:, 0]
dv2_miou = data[:, 1]
dv2_std = data[:, 3]

rf_miou = data[:, 2]
rf_std = data[:, 4]
data

array([[ 1.   ,  0.799,  0.349,  0.136,  0.102],
       [ 2.   ,  0.778,  0.38 ,  0.158,  0.102],
       [ 4.   ,  0.799,  0.393,  0.141,  0.103],
       [ 8.   ,  0.827,  0.398,  0.126,  0.112],
       [16.   ,  0.858,  0.407,  0.115,  0.126]])

In [84]:
def format_plot(ax, title: str, xlabel: str, ylabel: str, title_fontsize: int, label_fontsize: int, tick_fontsize: int, legend_fontsize: int ) -> None:
    plt.rcParams["font.family"] = "serif"
    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_ylabel(ylabel, fontsize=label_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    ax.set_title(title, fontsize=title_fontsize)
    ax.legend(fontsize=legend_fontsize)

In [85]:
%%capture
fig = plt.figure(2) #axs = plt.subplots(1, 2)
fig.set_size_inches(16, 16)

RED = '#d00000'
BLUE = '#1e74fd'
plt.errorbar(x, dv2_miou, dv2_std, label="Deep Features", marker='.', lw=6, ms=30, ecolor=RED, color=RED)
plt.errorbar(x, rf_miou, rf_std, label="Classical Features", marker='.', lw=6, ms=30)
format_plot(fig.gca(), "", "# labelled micrographs", "Class-averaged mIoU", 20, 32, 24, 24)

RuntimeError: latex was not able to process the following string:
b'# labelled micrographs'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpf19ae0k9 17ec2e4b82d6f004008d5e7438e9c33aacd690a9bf90f6831ec9c3d96cab6609.tex

This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./17ec2e4b82d6f004008d5e7438e9c33aacd690a9bf90f6831ec9c3d96cab6609.tex
LaTeX2e <2020-02-02> patch level 2
L3 programming layer <2020-02-14>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2019/12/20 v1.4l Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/share/texmf/tex/latex/cm-super/type1ec.sty
(/usr/share/texlive/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(

RuntimeError: latex was not able to process the following string:
b'# labelled micrographs'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error --output-directory=tmpwjy883l8 17ec2e4b82d6f004008d5e7438e9c33aacd690a9bf90f6831ec9c3d96cab6609.tex

This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/Debian) (preloaded format=latex)
 restricted \write18 enabled.
entering extended mode
(./17ec2e4b82d6f004008d5e7438e9c33aacd690a9bf90f6831ec9c3d96cab6609.tex
LaTeX2e <2020-02-02> patch level 2
L3 programming layer <2020-02-14>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2019/12/20 v1.4l Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/type1cm/type1cm.sty)
(/usr/share/texmf/tex/latex/cm-super/type1ec.sty
(/usr/share/texlive/texmf-dist/tex/latex/base/t1cmr.fd))
(/usr/share/texlive/texmf-dist/tex/latex/base/inputenc.sty)
(